## Περίληψη

Μια ομάδα επιχειρήσεων εφοδιαστικής σχεδιάζει ένα τυχαιοποιημένο πείραμα πεδίου που συγκρίνει τρεις στρατηγικές δρομολόγησης οδηγών (στατική βάση αναφοράς, δυναμική ανακατεύθυνση, βελτιστοποιημένη με AI) σε τρεις περιοχές αποθηκών, με μέση καθυστέρηση παράδοσης (λεπτά) ως απόκριση. Η PROC GLMPOWER λαμβάνει ένα *ενδεικτικό* σύνολο δεδομένων εικαζόμενων μέσων όρων κελιών και επιλύει για τον συνολικό αριθμό οδηγών που απαιτούνται για τον εντοπισμό επιδράσεων στρατηγικής με ισχύ 80% και 90%, και στη συνέχεια χαρτογραφεί πώς διαβρώνεται η επιτεύξιμη ισχύς καθώς αυξάνεται η μεταβλητότητα από διαδρομή σε διαδρομή.

# Προσδιορισμός Μεγέθους Πειράματος Πεδίου Δρομολόγησης Οδηγών με την PROC GLMPOWER

## Περίληψη

Μια ομάδα επιχειρήσεων εφοδιαστικής πρόκειται να ξεκινήσει ένα τυχαιοποιημένο πείραμα πεδίου που συγκρίνει τρεις στρατηγικές δρομολόγησης οδηγών -- μια βάση αναφοράς **Static**, ένα σύστημα ανακατεύθυνσης **Dynamic**, και έναν σχεδιαστή **AI-Optimized** -- αναπτυγμένες σε τρεις περιοχές αποθηκών (Northeast, Midwest, West). Η απόκριση είναι η μέση **καθυστέρηση παράδοσης σε λεπτά**. Πριν δεσμεύσει χωρητικότητα στόλου στο πείραμα, η ομάδα πρέπει να απαντήσει: *πόσους οδηγούς χρειαζόμαστε για να εντοπίσουμε αξιόπιστα τη λειτουργική βελτίωση που αναμένουμε;*

Αυτό το σημειωματάριο χρησιμοποιεί την **PROC GLMPOWER** για να εκτελέσει προοπτική ανάλυση ισχύος και μεγέθους δείγματος για το γενικό γραμμικό μοντέλο πίσω από το πείραμα. Ξεκινώντας από ένα *ενδεικτικό* σύνολο δεδομένων εικαζόμενων μέσων όρων κελιών, (1) επιλύει για τη συνολική εγγραφή που απαιτείται για να φτάσει σε ισχύ 80% και 90% για τις συνολικές επιδράσεις στρατηγικής και περιοχής, (2) χαρτογραφεί πώς υποβαθμίζεται η επιτεύξιμη ισχύς καθώς αυξάνεται η μεταβλητότητα από διαδρομή σε διαδρομή, και (3) παράγει μια καμπύλη ισχύος για να υποστηρίξει την απόφαση εγγραφής.

> **Βασικό συμπέρασμα:** Με μια εύλογη τυπική απόκλιση σφάλματος 8 λεπτών, περίπου **63 οδηγοί** αποδίδουν ισχύ 80% και **83 οδηγοί** αποδίδουν ισχύ 90% για τον εντοπισμό επιδράσεων στρατηγικής δρομολόγησης -- αλλά αν η μεταβλητότητα καθυστέρησης πλησιάζει τα 10 λεπτά, ακόμη και 90 εγγεγραμμένοι οδηγοί δεν φτάνουν το 70% ισχύος, υπογραμμίζοντας την αξία στενών, καλά ενοργανωμένων διαδρομών.

---
## 1. Δημιουργία του Ενδεικτικού Σχεδιασμού

Το πρώτο βήμα κωδικοποιεί τη διάταξη του πειράματος και τη *εικαζόμενη* μέση καθυστέρηση της ομάδας για κάθε συνδυασμό στρατηγικής-δρομολόγησης x περιοχής-αποθήκης. Καθορίζουμε έναν σπόρο τυχαιότητας (seed) και προσθέτουμε αμελητέο θόρυβο (jitter) ώστε οι μέσοι όροι να φαίνονται ρεαλιστικοί ενώ διατηρείται η ισορροπημένη δομή 3x3. Το βάρος `cell_n` ίσο με 1 σε κάθε κελί λέει στην GLMPOWER ότι ο σχεδιασμός είναι ισορροπημένος.

In [1]:
/* ================================================================
   Δημιουργία του ενδεικτικού συνόλου δεδομένων εικαζόμενων μέσων
   καθυστερήσεων. Μία γραμμή ανά κελί σχεδιασμού στρατηγικής-δρομολόγησης
   x περιοχής-αποθήκης.
   ================================================================ */
ΔΕΔΟΜΕΝΑ routing_trial;
   CALL streaminit(20260531);
   LENGTH routing_strategy $8 depot_region $9;
   ARRAY strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   ARRAY region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   ARRAY smean[3]     _temporary_ (18.0 14.5 11.0);   /* εικαζόμενοι μέσοι όροι στρατηγικής */
   ARRAY radj[3]      _temporary_ (1.5  0.0 -1.0);    /* περιφερειακές προσαρμογές (λεπτά)  */
   ΕΠΑΝΑΛΗΨΗ i = 1 ΕΩΣ 3;
      ΕΠΑΝΑΛΗΨΗ j = 1 ΕΩΣ 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         ΕΞΟΔΟΣ;
      ΤΕΛΟΣ;
   ΤΕΛΟΣ;
   ΑΦΑΙΡΕΣΗ i j;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=routing_trial ΕΤΙΚΕΤΑ noobs;
   ΜΕΤΑΒΛΗΤΗ routing_strategy depot_region mean_delay_min cell_n;
   ΕΤΙΚΕΤΑ routing_strategy="Στρατηγική Δρομολόγησης" depot_region="Περιοχή Αποθήκης"
         mean_delay_min="Μέση Καθυστέρηση (λεπτά)" cell_n="Μέγεθος Κελιού (n)";
   TITLE "Ενδεικτικοί Μέσοι Όροι Κελιών: Εικαζόμενη Καθυστέρηση Παράδοσης (λεπτά)";
ΕΚΤΕΛΕΣΗ;

                        Ενδεικτικοί Μέσοι Όροι Κελιών: Εικαζόμενη Καθυστέρηση Παράδοσης (λεπτά)                         

                      Στρατηγική Δρομολόγησης                 Περιοχή Αποθήκης                      Μέση Καθυστέρηση (λεπτά)               Μέγεθος Κελιού (n)
Static                                         Northeast                                                        19.687507651                                1
Static                                         Midwest                                                         17.9871067398                                1
Static                                         West                                                            16.8240210086                                1
Dynamic                                        Northeast                                                       15.9537535365                                1
Dynamic                                        Midwest                                  


NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Μέγεθος Δείγματος για τον Συνολικό Σχεδιασμό

Έχοντας τον σχεδιασμό στα χέρια μας, ζητάμε από την GLMPOWER να **επιλύσει για το συνολικό μέγεθος δείγματος** (`NTOTAL = .`) με ισχύ 80% και 90%. Η δήλωση `MODEL` προσδιορίζει μια διάταξη δύο κατευθύνσεων με αλληλεπίδραση (`routing_strategy*depot_region`), η `WEIGHT cell_n` δηλώνει τα ισορροπημένα βάρη προφίλ, και η `STDDEV = 8` είναι η υποτιθέμενη τετραγωνική ρίζα του μέσου τετραγωνικού σφάλματος της καθυστέρησης παράδοσης. Η `NFRACTIONAL` επιτρέπει στη διαδικασία να αναφέρει ακριβείς κλασματικές μετρήσεις πριν από τη στρογγυλοποίηση προς τα πάνω.

Καταχωρούμε επίσης εκ των προτέρων τρεις προγραμματισμένες συγκρίσεις `CONTRAST` -- AI-Opt έναντι Static, Dynamic έναντι Static, και οποιαδήποτε ανακατεύθυνση έναντι Static -- οι οποίες τεκμηριώνουν τις λειτουργικά ουσιαστικές υποθέσεις που έχει σχεδιαστεί να ελέγξει το πείραμα.

In [2]:
/* ================================================================
   Επίλυση για τον συνολικό αριθμό οδηγών που απαιτούνται για να
   φτάσουμε ισχύ 80% και 90% για τις επιδράσεις στρατηγικής-δρομολόγησης
   και περιοχής-αποθήκης.
   ================================================================ */
ΔΙΑΔΙΚΑΣΙΑ glmpower ΔΕΔΟΜΕΝΑ=routing_trial;
   ΚΛΑΣΗ routing_strategy depot_region;
   ΜΟΝΤΕΛΟ mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   ΒΑΡΟΣ cell_n;
   ΕΤΙΚΕΤΑ routing_strategy="Στρατηγική Δρομολόγησης" depot_region="Περιοχή Αποθήκης"
         mean_delay_min="Μέση Καθυστέρηση (λεπτά)";
   CONTRAST "AI-Opt έναντι Static"      routing_strategy -1  0  1;
   CONTRAST "Dynamic έναντι Static"     routing_strategy -1  1  0;
   CONTRAST "Οποιαδήποτε Ανακατεύθυνση έναντι Static" routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   TITLE "Μέγεθος Δείγματος για τον Εντοπισμό Επιδράσεων της Στρατηγικής Δρομολόγησης στην Καθυστέρηση";
ΕΚΤΕΛΕΣΗ;

                        Ενδεικτικοί Μέσοι Όροι Κελιών: Εικαζόμενη Καθυστέρηση Παράδοσης (λεπτά)                         

The GLMPOWER Procedure


                                     Fixed Scenario Elements                                     

Item                Value                                                                        
------------------  -----------------------------------------------------------------------------
Dependent Variable  Μέση Καθυστέρηση (λεπτά)                                                     
Source              Στρατηγική Δρομολόγησης                                                      
Source              Περιοχή Αποθήκης                                                             
Source              Στρατηγική Δρομολόγησης*Περιοχή Αποθήκης                                     

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0


NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Ευαισθησία Ισχύος στη Μεταβλητότητα και το Μέγεθος Πειράματος

Η απάντηση για το μέγεθος δείγματος εξαρτάται από την υποτιθέμενη τυπική απόκλιση σφάλματος, η οποία σπάνια είναι γνωστή με ακρίβεια εκ των προτέρων. Εδώ αντιστρέφουμε το ερώτημα: **καθορίζουμε** αρκετά υποψήφια συνολικά μεγέθη εγγραφής (`NTOTAL = 45 90 180`) και **επιλύουμε για την επιτευχθείσα ισχύ** (`POWER = .`) σε ένα πλέγμα εύλογων τυπικών αποκλίσεων καθυστέρησης (6, 8 και 10 λεπτά). Ο προκύπτων πίνακας είναι ένας χάρτης ευαισθησίας -- δείχνει πόσο ανθεκτικό είναι κάθε σχέδιο εγγραφής αν η πραγματική μεταβλητότητα διαδρομής αποδειχθεί υψηλότερη από την αναμενόμενη.

In [3]:
/* ================================================================
   Πλέγμα ευαισθησίας: επιτευχθείσα ισχύς σε υποψήφια μεγέθη πειράματος
   και εύλογες τυπικές αποκλίσεις καθυστέρησης.
   ================================================================ */
ΔΙΑΔΙΚΑΣΙΑ glmpower ΔΕΔΟΜΕΝΑ=routing_trial;
   ΚΛΑΣΗ routing_strategy depot_region;
   ΜΟΝΤΕΛΟ mean_delay_min = routing_strategy depot_region;
   ΒΑΡΟΣ cell_n;
   ΕΤΙΚΕΤΑ routing_strategy="Στρατηγική Δρομολόγησης" depot_region="Περιοχή Αποθήκης"
         mean_delay_min="Μέση Καθυστέρηση (λεπτά)";
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   TITLE "Επιτευχθείσα Ισχύς σε Σενάρια Μεταβλητότητας και Μεγέθους Δοκιμής";
ΕΚΤΕΛΕΣΗ;

                        Ενδεικτικοί Μέσοι Όροι Κελιών: Εικαζόμενη Καθυστέρηση Παράδοσης (λεπτά)                         

The GLMPOWER Procedure


                     Fixed Scenario Elements                     

Item                Value                                        
------------------  ---------------------------------------------
Dependent Variable  Μέση Καθυστέρηση (λεπτά)                     
Source              Στρατηγική Δρομολόγησης                      
Source              Περιοχή Αποθήκης                             

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       


NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Καμπύλη Ισχύος για την Απόφαση Εγγραφής

Τέλος, χαράσσουμε μια **καμπύλη ισχύος** -- επιτευχθείσα ισχύς καθώς η συνολική εγγραφή αυξάνεται από 30 σε 270 οδηγούς σε βήματα των 30 -- για το μοντέλο στρατηγικής-συν-περιοχής στην αναμενόμενη τυπική απόκλιση 8 λεπτών. Η επίλυση της `POWER = .` σε αυτό το πλέγμα εγγραφής παράγει την καμπύλη ως μια πινακοποιημένη σειρά `N Total` έναντι `Power`, ώστε να μπορούμε να διαβάσουμε την εγγραφή στην οποία διασχίζεται καθένας από τους συμβατικούς στόχους 80% και 90%.

In [4]:
/* ================================================================
   Καμπύλη ισχύος: επιτευχθείσα ισχύς έναντι συνολικού αριθμού
   εγγεγραμμένων οδηγών, από 30 έως 270 σε βήματα των 30 στην
   αναμενόμενη τυπική απόκλιση 8 λεπτών.
   ================================================================ */
ΔΙΑΔΙΚΑΣΙΑ glmpower ΔΕΔΟΜΕΝΑ=routing_trial;
   ΚΛΑΣΗ routing_strategy depot_region;
   ΜΟΝΤΕΛΟ mean_delay_min = routing_strategy depot_region;
   ΒΑΡΟΣ cell_n;
   ΕΤΙΚΕΤΑ routing_strategy="Στρατηγική Δρομολόγησης" depot_region="Περιοχή Αποθήκης"
         mean_delay_min="Μέση Καθυστέρηση (λεπτά)";
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   TITLE "Καμπύλη Ισχύος: Επιτευχθείσα Ισχύς έναντι Συνολικού Αριθμού Εγγεγραμμένων Οδηγών";
ΕΚΤΕΛΕΣΗ;

                        Ενδεικτικοί Μέσοι Όροι Κελιών: Εικαζόμενη Καθυστέρηση Παράδοσης (λεπτά)                         

The GLMPOWER Procedure


                     Fixed Scenario Elements                     

Item                Value                                        
------------------  ---------------------------------------------
Dependent Variable  Μέση Καθυστέρηση (λεπτά)                     
Source              Στρατηγική Δρομολόγησης                      
Source              Περιοχή Αποθήκης                             

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       


NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Ερμηνεία και Σύσταση

Η ανάλυση δίνει στην ομάδα επιχειρήσεων ένα τεκμηριωμένο σχέδιο εγγραφής:

- **Βασική διαστασιολόγηση (Ενότητα 2).** Υποθέτοντας τυπική απόκλιση καταλοίπων 8 λεπτών, το πλήρες μοντέλο δύο κατευθύνσεων (στρατηγική, περιοχή και η αλληλεπίδρασή τους) φτάνει **ισχύ 80% με 63 οδηγούς** και **ισχύ 90% με 83 οδηγούς**. Στρογγυλοποιώντας προς τα πάνω για απώλειες, ένας στόχος εγγραφής κοντά στους **90 οδηγούς** ξεπερνά άνετα το όριο του 90%.

- **Η ευαισθησία έχει σημασία (Ενότητα 3).** Η ισχύς είναι εξαιρετικά ευαίσθητη στη μεταβλητότητα καθυστέρησης. Με 90 οδηγούς, η επιτευχθείσα ισχύς πέφτει από ~99% (SD=6) σε ~87% (SD=8) και σε ~68% (SD=10). Ένα πιλοτικό πρόγραμμα 45 οδηγών είναι επαρκές μόνο αν η μεταβλητότητα είναι χαμηλή (~81% ισχύς στο SD=6) και είναι σοβαρά υποδιαστασιολογημένο (~37%) αν το SD φτάσει στο 10. Η πρακτική συνέπεια: η επένδυση σε συνεπείς, καλά ενοργανωμένες διαδρομές για τον περιορισμό της μεταβλητότητας είναι εξίσου πολύτιμη με την προσθήκη οδηγών.

- **Η καμπύλη ισχύος (Ενότητα 4).** Χαραγμένη για το μοντέλο στρατηγικής-συν-περιοχής (χωρίς όρο αλληλεπίδρασης, ο φακός που χρησιμοποιήθηκε για τη σάρωση ευαισθησίας), η επιτευχθείσα ισχύς ανεβαίνει από 0.37 στους 30 οδηγούς σε 0.69 στους 60, 0.87 στους 90 και 0.95 στους 120, ισοπεδώνοντας πάνω από 0.99 στους 180. Διαβάζοντας την καμπύλη έναντι των στόχων, η ισχύς 80% φτάνει κοντά στους 77 οδηγούς και η 90% κοντά στους 99 -- ελαφρώς υψηλότερα από τη διαστασιολόγηση πλήρους μοντέλου στην Ενότητα 2, επειδή η αφαίρεση του όρου αλληλεπίδρασης διασκορπίζει την ίδια επίδραση σε λιγότερους βαθμούς ελευθερίας του μοντέλου.

**Σύσταση:** Εγγράψτε περίπου **90 οδηγούς** (30 ανά στρατηγική δρομολόγησης, ισορροπημένους στις τρεις περιοχές αποθηκών). Αυτό ξεπερνά το 90% ισχύος για το πλήρες μοντέλο υπό την αναμενόμενη τυπική απόκλιση 8 λεπτών, διατηρεί ~87% ισχύ ακόμη και στην πιο συντηρητική καμπύλη μειωμένου μοντέλου, και κρατά το πείραμα αρκετά μικρό ώστε να εκτελεστεί μέσα σε ένα μόνο λειτουργικό τρίμηνο.

*Σημείωση:* Η GLMPOWER αναλύει τον εικαζόμενο *σχεδιασμό*, όχι παρατηρηθέντα αποτελέσματα -- επομένως η αξιοπιστία αυτών των αριθμών βασίζεται στους εικαζόμενους μέσους όρους και την τυπική απόκλιση. Οι ομάδες θα πρέπει να επανεξετάσουν τη διαστασιολόγηση μόλις ένα σύντομο πιλοτικό πρόγραμμα αποδώσει μια εμπειρική εκτίμηση της μεταβλητότητας καθυστέρησης παράδοσης.